In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
print(os.environ.get('HF_ENDPOINT'))   # 应输出 https://hf-mirror.com

https://hf-mirror.com


In [2]:
model_name = "./TinyLlama-1.1B-Chat-v1.0"   # 根据实际路径调整
tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16,  local_files_only=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"模型已加载到 {device}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

模型已加载到 cuda


In [3]:
dataset = load_dataset(
    "json",
    data_files="./dataset/alpaca-data-gpt4-chinese/Alpaca_data_gpt4_zh.jsonl",
    split="train"
)
dataset = dataset.select(range(2000))
print(dataset)

Dataset({
    features: ['instruction_zh', 'input_zh', 'output_zh', 'instruction', 'input', 'output'],
    num_rows: 2000
})


In [4]:
def format_instruction(example):
    instruction = example["instruction_zh"]
    input_text = example.get("input_zh", "")
    output = example["output_zh"]
    if input_text:
        text = f"### 指令：{instruction}\n### 输入：{input_text}\n### 回答：{output}"
    else:
        text = f"### 指令：{instruction}\n### 回答：{output}"
    return {"text": text}

dataset = dataset.map(format_instruction)
print(dataset[0]["text"])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

### 指令：给出三个保持健康的小贴士。
### 回答：1. 饮食要均衡且富有营养：确保你的餐食包含各种水果、蔬菜、瘦肉、全谷物和健康脂肪。这有助于为身体提供必要的营养，使其发挥最佳功能，并有助于预防慢性疾病。2. 经常参加体育锻炼：锻炼对于保持强壮的骨骼、肌肉和心血管健康至关重要。每周至少要进行150分钟的中等有氧运动或75分钟的剧烈运动。3. 获得足够的睡眠：获得足够的高质量睡眠对身体和心理健康至关重要。它有助于调节情绪，提高认知功能，并支持健康的生长和免疫功能。每晚睡眠目标为7-9小时。


In [5]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)
tokenized_dataset = tokenized_dataset.select_columns(["input_ids", "attention_mask"])

print("数据集大小:", len(tokenized_dataset))
print("第一个样本:", tokenized_dataset[0])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

数据集大小: 2000
第一个样本: {'input_ids': [1, 835, 29871, 31084, 31650, 30383, 31999, 30544, 30457, 30502, 30982, 31695, 31863, 31577, 30210, 30446, 235, 183, 183, 30927, 30267, 13, 2277, 29937, 29871, 30742, 234, 176, 151, 30383, 29896, 29889, 29871, 236, 168, 177, 31855, 30698, 232, 160, 138, 235, 164, 164, 231, 187, 151, 31730, 30417, 235, 147, 168, 232, 136, 190, 30383, 31835, 30982, 30919, 30210, 236, 167, 147, 31855, 31473, 232, 147, 174, 232, 147, 135, 31893, 30716, 30801, 30330, 235, 151, 175, 31854, 30330, 234, 155, 169, 235, 133, 140, 30330, 30753, 31112, 30834, 30503, 31863, 31577, 235, 135, 133, 235, 133, 173, 30267, 30810, 30417, 31931, 30909, 30573, 31687, 30988, 31302, 231, 193, 158, 31641, 30698, 30210, 235, 147, 168, 232, 136, 190, 30214, 30785, 31149, 30910, 233, 143, 168, 30878, 231, 192, 182, 31134, 30815, 30214, 31666, 30417, 31931, 30909, 236, 165, 135, 236, 155, 181, 233, 136, 165, 30952, 234, 153, 193, 234, 154, 136, 30267, 29906, 29889, 29871, 31412, 31190, 31125, 30666

In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [7]:
from transformers import TrainingArguments, Trainer

def collate_fn(data):
    input_ids = torch.tensor([d["input_ids"] for d in data], dtype=torch.long, device=device)
    attention_mask = torch.tensor([d["attention_mask"] for d in data], dtype=torch.long, device=device)
    labels = torch.tensor([d["input_ids"] for d in data], dtype=torch.long, device=device)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

training_args = TrainingArguments(
    output_dir="./tinyllama-lora-gpu",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=100,
    learning_rate=1e-4,
    fp16=True,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to="none",
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=collate_fn,
)

trainer.train()

Step,Training Loss
10,6.106282
20,1.155747
30,0.829935
40,0.778393
50,0.734636
60,0.697183
70,0.704813
80,0.713868
90,0.686515
100,0.702748


TrainOutput(global_step=250, training_loss=0.926294807434082, metrics={'train_runtime': 202.3941, 'train_samples_per_second': 19.763, 'train_steps_per_second': 1.235, 'total_flos': 1.2767452987392e+16, 'train_loss': 0.926294807434082, 'epoch': 2.0})

In [8]:
from transformers import pipeline
# 1. 保存 LoRA 权重
lora_save_path = "./tuning_cache//tinyllama-lora-gpu-final-3"
model.save_pretrained(lora_save_path)
tokenizer.save_pretrained(lora_save_path)
print("✅ LoRA 权重已保存")

# 2. 可选：合并并保存完整模型
merge = False   # 设为 False 则只测试 LoRA 模型
if merge:
    # 重新加载基础模型（需与训练时使用的模型相同）
    base_model = AutoModelForCausalLM.from_pretrained(
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        torch_dtype=torch.float16,
        device_map="auto"
    )
    # 加载 LoRA 并合并
    merged_model = PeftModel.from_pretrained(base_model, lora_save_path).merge_and_unload()
    merged_model.save_pretrained("./tinyllama-merged")
    tokenizer.save_pretrained("./tinyllama-merged")
    print("✅ 完整模型已保存到 ./tinyllama-merged")
    test_model = merged_model
else:
    test_model = model   # 直接使用训练后的 PeftModel

# 3. 推理测试
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipe = pipeline(
    "text-generation",
    model=test_model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1  # 0 表示第一个 GPU
)

prompt = "### 指令：解释什么是深度学习\n### 回答："
output = pipe(
    prompt,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.6,           # 降低随机性
    top_p=0.85,
    repetition_penalty=1.1     # 增加重复惩罚
)
print("\n生成结果：")
print(output[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'repetition_penalty', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ LoRA 权重已保存

生成结果：
### 指令：解释什么是深度学习
### 回答：深度学习（Deep Learning）是一种在数据中进行决策的机器学习方法，其特点是将整个过程从最初的基本组件到最后的分类和回归问题转化为具有复杂性的模型。这些模型通常由多层神经网络（NN）构建，并利用大量的数据来训练。深度学习技术使用神经网络来解决复杂问题，包括协同过滤、图像分类、文本分类、语言模型等等。它们可以被应用于广告�����
